In [7]:
import requests
import pandas as pd

#한 번에 최대 1,000건 까지 가능
def make_response(numOfRows, startDt, endDt): #데이터 개수, 시작일자(YYYYMMDD), 끝일자
    SERVICE_KEY = "MFtQSWmriyJaWcgKOVr9Xuw2Gq%2BUhQa9dcGdEGDMK5pvvdptbr8Lc39CI5pW0xf4lgvwv7HzpzmyxDv4%2BG8KsA%3D%3D"

    url = (
        "http://apis.data.go.kr/1360000/AsosHourlyInfoService/getWthrDataList"
        f"?serviceKey={SERVICE_KEY}" # encoding 인증 키
        f"&numOfRows={numOfRows}&pageNo=1" # 한 번에 받을 데이터 개수 (24시간, 24개), 페이지 번호
        "&dataType=JSON" # 응답 형식
        "&dataCd=ASOS&dateCd=HR" # 자료 코드(ASOS), 날짜 코드(HR)
        f"&startDt={startDt}&startHh=00" # 조회 시작일, 조회 시작 시각
        f"&endDt={endDt}&endHh=23" # 조회 종료일, 조회 종료 시각
        "&stnIds=101" # 관측 지점 번호
    )

    return requests.get(url)

response1 = make_response(31*24, 20240101, 20240131) #1월
response2 = make_response(29*24, 20240201, 20240229) #2월
response3 = make_response(31*24, 20240301, 20240331) #3월

print("1월 상태코드:", response1.status_code)
print(response1.text[:300])

print("2월 상태코드:", response2.status_code)
print(response2.text[:300])

print("3월 상태코드:", response3.status_code)
print(response3.text[:300])

1월 상태코드: 200
{"response":{"header":{"resultCode":"00","resultMsg":"NORMAL_SERVICE"},"body":{"dataType":"JSON","items":{"item":[{"tm":"2024-01-01 00:00","rnum":"1","stnId":"101","stnNm":"춘천","ta":"0.8","taQcflg":"","rn":"","rnQcflg":"9","ws":"1.6","wsQcflg":"","wd":"320","wdQcflg":"","hm":"97","hmQcflg":"","pv":"
2월 상태코드: 200
{"response":{"header":{"resultCode":"00","resultMsg":"NORMAL_SERVICE"},"body":{"dataType":"JSON","items":{"item":[{"tm":"2024-02-01 00:00","rnum":"1","stnId":"101","stnNm":"춘천","ta":"-3.1","taQcflg":"","rn":"","rnQcflg":"","ws":"0.2","wsQcflg":"","wd":"0","wdQcflg":"","hm":"84","hmQcflg":"","pv":"4.
3월 상태코드: 200
{"response":{"header":{"resultCode":"00","resultMsg":"NORMAL_SERVICE"},"body":{"dataType":"JSON","items":{"item":[{"tm":"2024-03-01 00:00","rnum":"1","stnId":"101","stnNm":"춘천","ta":"2.2","taQcflg":"","rn":"","rnQcflg":"","ws":"1.9","wsQcflg":"","wd":"270","wdQcflg":"","hm":"48","hmQcflg":"","pv":"3


In [8]:
import pandas as pd

def make_df(resp):
    data = resp.json()

    # 응답 구조 확인
    items = data["response"]["body"]["items"]["item"]

    df = pd.DataFrame(items)

    # 프로젝트에서 쓸 컬럼만
    df = df[["tm", "ta", "hm", "ws"]].rename(columns={
        "tm": "timestamp",
        "ta": "outdoor_temp_c",        # 기온 (°C)
        "hm": "outdoor_humidity",    # 습도 (%)
        "ws": "outdoor_wind_speed",  # 풍속 (m/s)
    })

    df["outdoor_temp_c"] = pd.to_numeric(df["outdoor_temp_c"], errors="coerce")
    df["outdoor_humidity"] = pd.to_numeric(df["outdoor_humidity"], errors="coerce")
    df["outdoor_wind_speed"] = pd.to_numeric(df["outdoor_wind_speed"], errors="coerce")

    return df

df1 = make_df(response1)
df2 = make_df(response2)
df3 = make_df(response3)

print("==1월==")
print(df1)
print("==2월==")
print(df2)
print("==3월==")
print(df3)

==1월==
            timestamp  outdoor_temp_c  outdoor_humidity  outdoor_wind_speed
0    2024-01-01 00:00             0.8                97                 1.6
1    2024-01-01 01:00             1.0                97                 1.1
2    2024-01-01 02:00             0.9                96                 0.4
3    2024-01-01 03:00             0.9                96                 0.4
4    2024-01-01 04:00             0.7                95                 0.6
..                ...             ...               ...                 ...
739  2024-01-31 19:00             4.1                48                 1.1
740  2024-01-31 20:00             1.0                59                 0.3
741  2024-01-31 21:00            -0.5                71                 0.7
742  2024-01-31 22:00            -1.6                73                 0.0
743  2024-01-31 23:00            -2.5                80                 0.0

[744 rows x 4 columns]
==2월==
            timestamp  outdoor_temp_c  outdoor_hum

In [9]:
import numpy as np
import matplotlib.pyplot as plt

def preprocess_df(df):
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    return df


pre1_df1 = preprocess_df(df1)
pre1_df2 = preprocess_df(df2)
pre1_df3 = preprocess_df(df3)

def process_null(df):
    print("=== 처리 전 결측치 ===")
    print(df.isnull().sum())

    # 기온: 선형 보간 (앞뒤 값으로 채우기, 최대 3시간 연속까지만)
    df["outdoor_temp_c"] = df["outdoor_temp_c"].interpolate(method="linear", limit=3)

    # 습도: 선형 보간
    df["outdoor_humidity"] = df["outdoor_humidity"].interpolate(method="linear", limit=3)

    # 풍속: 0으로 채우기 (바람 없는 걸로 간주)
    df["outdoor_wind_speed"] = df["outdoor_wind_speed"].fillna(0)

    # 그래도 남은 결측치 (3시간 넘게 연속 결측) → 앞/뒤 값으로 채우기
    df = df.ffill().bfill()

    print("\n=== 처리 후 결측치 ===")
    print(df.isnull().sum())

    return df

pre2_df1 = process_null(pre1_df1)
pre2_df2 = process_null(pre1_df2)
pre2_df3 = process_null(pre1_df3)

def process_outlier(df):
    outlier_temp = df["outdoor_temp_c"][(df["outdoor_temp_c"] < -30) | (df["outdoor_temp_c"] > 50)]
    print(f"기온 이상치: {len(outlier_temp)}개")
    print(outlier_temp)

    df["outdoor_temp_c"] = df["outdoor_temp_c"].where(
        df["outdoor_temp_c"].between(-30, 50)  # 범위 밖은 NaN으로
    ).interpolate(method="linear")    # NaN은 보간으로 채우기

    # 습도: 0~100% 범위 강제
    df["outdoor_humidity"] = df["outdoor_humidity"].clip(0, 100)

    # 풍속: 음수 불가
    df["outdoor_wind_speed"] = df["outdoor_wind_speed"].clip(0, None)
    return df


pre3_df1 = process_outlier(pre2_df1)
pre3_df2 = process_outlier(pre2_df2)
pre3_df3 = process_outlier(pre2_df3)
print(pre3_df1)
print(pre3_df2)
print(pre3_df3)

=== 처리 전 결측치 ===
timestamp             0
outdoor_temp_c        0
outdoor_humidity      0
outdoor_wind_speed    0
dtype: int64

=== 처리 후 결측치 ===
timestamp             0
outdoor_temp_c        0
outdoor_humidity      0
outdoor_wind_speed    0
dtype: int64
=== 처리 전 결측치 ===
timestamp             0
outdoor_temp_c        0
outdoor_humidity      0
outdoor_wind_speed    0
dtype: int64

=== 처리 후 결측치 ===
timestamp             0
outdoor_temp_c        0
outdoor_humidity      0
outdoor_wind_speed    0
dtype: int64
=== 처리 전 결측치 ===
timestamp             0
outdoor_temp_c        0
outdoor_humidity      0
outdoor_wind_speed    0
dtype: int64

=== 처리 후 결측치 ===
timestamp             0
outdoor_temp_c        0
outdoor_humidity      0
outdoor_wind_speed    0
dtype: int64
기온 이상치: 0개
Series([], Name: outdoor_temp_c, dtype: float64)
기온 이상치: 0개
Series([], Name: outdoor_temp_c, dtype: float64)
기온 이상치: 0개
Series([], Name: outdoor_temp_c, dtype: float64)
              timestamp  outdoor_temp_c  outdoor_humidity  ou

In [10]:
#import os
#os.makedirs("processed", exist_ok=True)

df_final = pd.concat([pre3_df1, pre3_df2, pre3_df3], ignore_index=True)

df_final.to_parquet("processed/weather.parquet", index=False)
print("저장 완료! →  processed/weather.parquet")
df_final.to_csv("processed/weather.csv", index=False)
print("저장 완료! →  processed/weather.csv")

print(df_final.dtypes)

# 잘 저장됐는지 확인
df_check = pd.read_parquet("processed/weather.parquet")
df_check2 = pd.read_csv("processed/weather.csv")
print(f"불러오기 확인: {df_check.shape}")
df_check.head()

저장 완료! →  processed/weather.parquet
저장 완료! →  processed/weather.csv
timestamp             datetime64[ns]
outdoor_temp_c               float64
outdoor_humidity               int64
outdoor_wind_speed           float64
dtype: object
불러오기 확인: (2184, 4)


,timestamp,outdoor_temp_c,outdoor_humidity,outdoor_wind_speed
0,2024-01-01 00:00:00,0.8,97,1.6
1,2024-01-01 01:00:00,1.0,97,1.1
2,2024-01-01 02:00:00,0.9,96,0.4
3,2024-01-01 03:00:00,0.9,96,0.4
4,2024-01-01 04:00:00,0.7,95,0.6
